<a href="https://colab.research.google.com/github/prodigal94/food-safe-bots/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Initialize Spark with optimizations for large data
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("BDA_Project") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Re-attempt reading the file locally
df = spark.read.csv(output, header=True, inferSchema=True)
display(df.limit(5).toPandas())

,Flag,Description
0,A,Official figure
1,E,Estimated value
2,I,Value imputed by a receiving agency
3,X,Figure from external organization


In [ ]:
# Update to use the file downloaded via gdown
# 'output' was defined in the gdown cell (31c82f7e)
df = spark.read.csv(output, header=True, inferSchema=True)
print(f"Successfully loaded: {output}")

Successfully loaded: Trade_Data_Folder/Trade_CropsLivestock_E_Flags.csv


In [ ]:
import os
# List files in the CSV directory to verify the filename
csv_dir = '/content/drive/MyDrive/CSV/'
if os.path.exists(csv_dir):
    print("Files in directory:")
    for f in os.listdir(csv_dir):
        print(f)
else:
    print(f"Directory not found: {csv_dir}")

Directory not found: /content/drive/MyDrive/CSV/


In [ ]:
df_before = df  # This is just a reference, not a copy

In [ ]:
# Cache the dataframe to avoid recomputation
df.cache()

total_records = df.count()
print(f"Total records: {total_records:,}")
print(f"Total columns: {len(df.columns)}")

Total records: 4
Total columns: 2


In [ ]:
print("\n--- Dataset Overview (BEFORE) ---")
# Check rows with missing values BEFORE
print("\n--- Rows with Missing Values (BEFORE) ---")

# Count rows that have ANY missing value
rows_with_missing = df.filter(
    F.greatest(*[F.when(F.col(c).isNull(), 1).otherwise(0) for c in df.columns]) == 1
).count()

rows_complete = total_records - rows_with_missing
print(f"  Complete rows (no missing): {rows_complete:,} ({rows_complete/total_records*100:.2f}%)")
print(f"  Rows with missing values: {rows_with_missing:,} ({rows_with_missing/total_records*100:.2f}%)")


--- Dataset Overview (BEFORE) ---

--- Rows with Missing Values (BEFORE) ---
  Complete rows (no missing): 4 (100.00%)
  Rows with missing values: 0 (0.00%)


In [ ]:
# Explicitly load the main data file from the downloaded folder
main_data_path = 'Trade_Data_Folder/Trade_CropsLivestock_E_All_Data_(Normalized).csv'
df = spark.read.csv(main_data_path, header=True, inferSchema=True)

# Now get year range
min_year = df.select(F.min('Year')).collect()[0][0]
max_year = df.select(F.max('Year')).collect()[0][0]
print(f"Year range: {min_year} - {max_year}")

Year range: 1961 - 2024


In [ ]:
# Check duplicates BEFORE
print("\n--- Duplicate Records (BEFORE) ---")
duplicates_before = total_records - df.distinct().count()
print(f"Duplicate rows: {duplicates_before:,}")


--- Duplicate Records (BEFORE) ---
Duplicate rows: -17,270,627


In [ ]:
# EDA 1: TOTAL TRADE ITEMS (ALL PRODUCTS TOGETHER - RECORD COUNTS ONLY)
print("\n" + "=" * 60)
print("EDA 1: COUNT FOR EACH TRADE ITEM (ALL PRODUCTS)")
print("=" * 60)

# Show total distinct trade items
distinct_items = df_before.select("Item").distinct().count()
print(f"\n TOTAL DISTINCT TRADE ITEMS: {distinct_items:,}")
print(f" TOTAL RECORDS: {df_before.count():,}")

# Count for each trade item (ALL items together) - NO VALUE
item_counts_before = df_before.groupBy("Item").agg(
    F.count("*").alias("Record_Count")
).orderBy(F.desc("Record_Count"))

print("\n TOP 20 ITEMS BY RECORD COUNT (ALL PRODUCTS TOGETHER):")
item_counts_before.show(20, truncate=False)

# Collect for visualization (top 15 for better view)
item_top_list = item_counts_before.limit(15).collect()
items_top = [row.Item for row in item_top_list]
counts_top = [row.Record_Count for row in item_top_list]

# VISUALIZATION
plt.figure(figsize=(12, 8))

plt.barh(items_top, counts_top, color='steelblue', edgecolor='black')
plt.title('Top 15 Trade Items by Record Count\n(All Crops & Livestock Products Together)',
          fontsize=14, fontweight='bold')
plt.xlabel('Number of Records', fontsize=12)
plt.ylabel('Trade Item', fontsize=12)

# Add value labels
for i, v in enumerate(counts_top):
    plt.text(v + 1000, i, f'{v:,}', va='center', fontsize=9)

plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

# Simple summary
print("\n" + "=" * 60)
print("SUMMARY STATISTICS")
print("=" * 60)
print(f"Total Distinct Items: {distinct_items:,}")
print(f"Total Records: {df_before.count():,}")
print(f"Average Records per Item: {df_before.count() / distinct_items:,.1f}")

# Most and least frequent items
print(f"\nMost frequent item: {item_counts_before.first()['Item']} ({item_counts_before.first()['Record_Count']:,} records)")
least_frequent = item_counts_before.orderBy(F.asc("Record_Count")).filter(F.col("Record_Count") > 0).first()
print(f"Least frequent item: {least_frequent['Item']} ({least_frequent['Record_Count']:,} records)")

In [ ]:
# DATA CLEANING
print("\n" + "=" * 60)
print("DATA CLEANING")
print("=" * 60)

df = df.dropDuplicates()

df = df.na.fill("Unknown", subset=["Area", "Item", "Element", "Unit", "Flag", "Note"])

df = df.na.fill(0, subset=["Value"])

df = df.dropna(subset=["Year"])

df = df.select("Area", "Item", "Element", "Year", "Unit", "Value", "Flag", "Note")

df = df.orderBy("Area", "Item", "Year")

df_after = df

print("Rows after cleaning:", df_after.count())
print("Columns after cleaning:", len(df_after.columns))
df_after.show(5, truncate=False)


In [ ]:
print("\n--- Dataset Overview (AFTER) ---")
print("\n--- Rows with Missing Values (AFTER) ---")

total_records_after = df_after.count()

rows_with_missing_after = df_after.filter(
    F.greatest(*[F.when(F.col(c).isNull(), 1).otherwise(0) for c in df_after.columns]) == 1
).count()

rows_complete_after = total_records_after - rows_with_missing_after
print(f"  Complete rows (no missing): {rows_complete_after:,} ({rows_complete_after/total_records_after*100:.2f}%)")
print(f"  Rows with missing values: {rows_with_missing_after:,} ({rows_with_missing_after/total_records_after*100:.2f}%)")


In [ ]:
min_year_after = df_after.select(F.min('Year')).collect()[0][0]
max_year_after = df_after.select(F.max('Year')).collect()[0][0]
print(f"Year range after cleaning: {min_year_after} - {max_year_after}")


In [ ]:
# EDA 1: TOTAL TRADE ITEMS AFTER CLEANING
print("\n" + "=" * 60)
print("EDA 2: COUNT FOR EACH TRADE ITEM (AFTER CLEANING)")
print("=" * 60)

distinct_items_after = df_after.select("Item").distinct().count()
print(f"\nTOTAL DISTINCT TRADE ITEMS: {distinct_items_after:,}")
print(f"TOTAL RECORDS: {df_after.count():,}")

item_counts_after = df_after.groupBy("Item").agg(
    F.count("*").alias("Record_Count")
).orderBy(F.desc("Record_Count"))

print("\nTOP 20 ITEMS BY RECORD COUNT:")
item_counts_after.show(20, truncate=False)

item_top_list_after = item_counts_after.limit(15).collect()
items_top_after = [row.Item for row in item_top_list_after]
counts_top_after = [row.Record_Count for row in item_top_list_after]

plt.figure(figsize=(12, 8))
plt.barh(items_top_after, counts_top_after, color='darkgreen', edgecolor='black')
plt.title('Top 15 Trade Items by Record Count\n(After Cleaning)', fontsize=14, fontweight='bold')
plt.xlabel('Number of Records', fontsize=12)
plt.ylabel('Trade Item', fontsize=12)

for i, v in enumerate(counts_top_after):
    plt.text(v + 1000, i, f'{v:,}', va='center', fontsize=9)

plt.grid(axis='x', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()
